# 🛒 E-Commerce Sales Performance Analysis

**Author:** Tannu Kumari | IGDTUW Delhi | B.Tech ECE | Batch 2026  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Dataset:** E-Commerce Retail Sales Data (5000 orders, 12 months, 2023)  

---

This is a full EDA project on a retail sales dataset. I wanted to figure out things like — which category makes the most money, does giving discount actually hurt profit, which region is strongest, that kind of stuff. Let's go step by step.

## 1. Importing Libraries

Standard stuff — pandas for data, matplotlib and seaborn for charts, warnings off so output stays clean.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# chart styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor': '#FFFFFF',
})

COLORS = ['#2563EB', '#16A34A', '#DC2626', '#D97706', '#7C3AED', '#0891B2']

## 2. Loading the Dataset

I'm using a real retail sales dataset with 5000 orders across 5 product categories, 5 regions, and 3 customer segments — covers the full year 2023.  

First thing I always do is `.head()` to just *see* what the data looks like, then `.info()` to check dtypes and nulls.

In [ ]:
df = pd.read_csv('ecommerce_sales_data.csv', parse_dates=['order_date'])
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# quick check — any nulls?
df.isnull().sum()

## 3. Basic Dataset Overview

No nulls — clean dataset, nice. Now let me print out the key business numbers upfront so I know what I'm working with before going into charts.

In [ ]:
print('=' * 55)
print('   E-COMMERCE SALES PERFORMANCE ANALYSIS — 2023')
print('=' * 55)

print(f"""
📦 Dataset Overview
   Total Orders     : {len(df):,}
   Date Range       : {df['order_date'].min().date()} → {df['order_date'].max().date()}
   Categories       : {df['category'].nunique()}
   Regions          : {df['region'].nunique()}

💰 Business Summary
   Total Revenue    : ₹{df['revenue'].sum()/1e6:.2f} Lakhs
   Total Profit     : ₹{df['profit'].sum()/1e6:.2f} Lakhs
   Avg Profit Margin: {df['profit_margin'].mean():.1f}%
   Avg Order Value  : ₹{df['revenue'].mean():,.0f}
   Total Units Sold : {df['quantity'].sum():,}
""")

In [ ]:
# category-wise revenue — just to see the numbers before plotting
cat_rev = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print('Revenue by Category (₹ Lakhs):')
for cat, rev in cat_rev.items():
    print(f'  {cat:<20} ₹{rev/1e6:.2f}L')

## 4. Revenue & Profit Trends Over Time

I wanted to see how revenue and profit move across the months — whether there's any seasonal pattern or a spike during festive months. I also broke it down by quarter to see which quarter did best overall.  

I used a line chart with filled area for the monthly trend because it looks cleaner than a plain line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('E-Commerce Performance Dashboard 2023', fontsize=16, fontweight='bold', y=1.01)

# group by month
monthly = (
    df.groupby('month_num')
      .agg(revenue=('revenue', 'sum'), profit=('profit', 'sum'))
      .reset_index()
      .sort_values('month_num')
)
months_short = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly['label'] = months_short

# left — monthly trend
ax = axes[0]
ax.fill_between(monthly['label'], monthly['revenue']/1e6, alpha=0.3, color=COLORS[0])
ax.plot(monthly['label'], monthly['revenue']/1e6, 'o-', color=COLORS[0], lw=2.5, ms=6, label='Revenue')
ax.fill_between(monthly['label'], monthly['profit']/1e6, alpha=0.3, color=COLORS[1])
ax.plot(monthly['label'], monthly['profit']/1e6, 's--', color=COLORS[1], lw=2, ms=5, label='Profit')
ax.set_title('Monthly Revenue vs Profit (₹ Lakhs)')
ax.set_xlabel('Month')
ax.set_ylabel('Amount (₹ Lakhs)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('₹%.0fL'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# right — quarterly bar
ax2 = axes[1]
quarter = df.groupby('quarter')['revenue'].sum()
bars = ax2.bar(quarter.index, quarter.values/1e6, color=COLORS[:4], edgecolor='white', width=0.5)
ax2.set_title('Quarterly Revenue Breakdown')
ax2.set_xlabel('Quarter')
ax2.set_ylabel('Revenue (₹ Lakhs)')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'₹{bar.get_height():.0f}L', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('chart1_revenue_trends.png', bbox_inches='tight')
plt.show()

**What I found:** Revenue stays fairly consistent across the year — no major dip or festive spike surprisingly. Q2 is slightly better. The profit line follows revenue closely, which tells me the margin doesn't change much month-to-month.

## 5. Category & Region Analysis

This was one of the first questions I had — which category is actually making money, not just revenue? Because a high-revenue category can still have bad margins.  

I plotted both revenue and profit side-by-side as horizontal bars so the comparison is easy. For region, I went with a pie chart to show percentage share.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# left — category revenue vs profit
cat_data = df.groupby('category').agg(
    revenue=('revenue', 'sum'),
    profit=('profit', 'sum')
).sort_values('revenue', ascending=True)

ax = axes[0]
y = range(len(cat_data))
ax.barh(y, cat_data['revenue']/1e6, color=COLORS[0], alpha=0.8, label='Revenue', height=0.4)
ax.barh([i + 0.42 for i in y], cat_data['profit']/1e6, color=COLORS[1], alpha=0.8, label='Profit', height=0.4)
ax.set_yticks([i + 0.21 for i in y])
ax.set_yticklabels(cat_data.index)
ax.set_title('Revenue & Profit by Category')
ax.set_xlabel('Amount (₹ Lakhs)')
ax.legend()

# right — region pie
ax2 = axes[1]
reg_data = df.groupby('region')['revenue'].sum()
wedges, texts, autotexts = ax2.pie(
    reg_data, labels=reg_data.index, colors=COLORS[:5],
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
ax2.set_title('Revenue Share by Region')

plt.tight_layout()
plt.savefig('chart2_category_region.png', bbox_inches='tight')
plt.show()

**What I found:** Electronics completely dominates — it's way ahead of every other category in both revenue and profit. That makes sense since unit prices are so high. Books and Clothing are the weakest performers.  

For regions, the split is pretty even across all 5 — no single region is dominating, which is interesting. Central is slightly on top.

## 6. Discount Impact & Customer Segment Analysis

This one I was really curious about — does giving more discount actually kill profit margin? Because companies often give discounts to push sales but maybe it backfires.  

I binned discounts into 5 levels and plotted the average profit margin for each. Then I also looked at how each customer segment performs across categories.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# left — discount vs profit margin
ax = axes[0]
discount_groups = pd.cut(
    df['discount'],
    bins=[-0.01, 0, 0.05, 0.10, 0.15, 0.20],
    labels=['0%', '5%', '10%', '15%', '20%']
)
disc_profit = df.groupby(discount_groups)['profit_margin'].mean()
bar_colors = [COLORS[1] if v > 0 else COLORS[2] for v in disc_profit.values]
ax.bar(disc_profit.index, disc_profit.values, color=bar_colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Avg Profit Margin by Discount Level')
ax.set_xlabel('Discount %')
ax.set_ylabel('Avg Profit Margin (%)')
ax.set_ylim(disc_profit.min() - 5, disc_profit.max() + 5)

# right — segment × category revenue
ax2 = axes[1]
seg_data = df.groupby(['segment', 'category'])['revenue'].sum().unstack(fill_value=0)
seg_data.div(1e6).plot(kind='bar', ax=ax2, color=COLORS[:5], edgecolor='white', width=0.65)
ax2.set_title('Revenue by Customer Segment & Category')
ax2.set_xlabel('Segment')
ax2.set_ylabel('Revenue (₹ Lakhs)')
ax2.legend(title='Category', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=0)

plt.tight_layout()
plt.savefig('chart3_discount_segment.png', bbox_inches='tight')
plt.show()

**What I found:** Yes — discount definitely hurts margin. At 0% discount, average margin is ~43%. At 20% discount, it drops to ~28%. So the relationship is pretty clear. Probably shouldn't go above 10%.  

For segments — Corporate and Consumer are both strong. Electronics dominates in every segment, which confirms it's the key revenue driver across all customer types.

## 7. Heatmap — Which Day Does Each Category Sell Most?

I thought it'd be interesting to see if there's a pattern by day of week — like maybe Electronics sells more on weekends, or people buy Books on Mondays. I made a heatmap of revenue by weekday × category.  

Also added payment method chart next to it — which payment mode brings in most revenue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# left — heatmap weekday × category
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df.pivot_table(values='revenue', index='weekday', columns='category',
                        aggfunc='sum', fill_value=0)
pivot = pivot.reindex([d for d in order if d in pivot.index])

ax = axes[0]
sns.heatmap(pivot/1e6, ax=ax, cmap='Blues', annot=True, fmt='.0f',
            linewidths=0.5, cbar_kws={'label': 'Revenue (₹L)'})
ax.set_title('Revenue Heatmap — Day of Week × Category')
ax.set_xlabel('Category')
ax.set_ylabel('Weekday')

# right — payment method
ax2 = axes[1]
pay_data = df.groupby('payment')['revenue'].sum().sort_values(ascending=False)
bars = ax2.bar(pay_data.index, pay_data.values/1e6, color=COLORS, edgecolor='white')
ax2.set_title('Revenue by Payment Method')
ax2.set_xlabel('Payment Method')
ax2.set_ylabel('Revenue (₹ Lakhs)')
for bar in bars:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'₹{bar.get_height():.0f}L', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('chart4_heatmap_payment.png', bbox_inches='tight')
plt.show()

**What I found:** Electronics revenue is consistently high every single day — Wednesday is the peak day for it. Books and Clothing are too low to even show meaningful differences across days.  

For payment — COD leads which is interesting, followed by Credit Card and UPI. This tells me the customer base might have trust issues with prepaid payment or they're just habitual COD users.

## 8. Top 10 Products by Revenue

Finally I wanted to know — which specific products are bringing in the most money? I sorted by total revenue and picked the top 10. Color-coded them — blue for top 3, grey for middle, red for bottom 3 within top 10.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

top10 = df.groupby('product')['revenue'].sum().nlargest(10).sort_values()
colors_bar = [COLORS[0] if i >= 7 else COLORS[2] if i < 3 else '#6B7280' for i in range(10)]

bars = ax.barh(top10.index, top10.values/1e6, color=colors_bar, edgecolor='white')
ax.set_title('Top 10 Products by Revenue')
ax.set_xlabel('Revenue (₹ Lakhs)')
for bar in bars:
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'₹{bar.get_width():.1f}L', va='center', fontsize=9)

ax.legend(handles=[
    plt.Rectangle((0,0),1,1, color=COLORS[0], label='Top 3'),
    plt.Rectangle((0,0),1,1, color='#6B7280', label='Mid'),
    plt.Rectangle((0,0),1,1, color=COLORS[2], label='Bottom 3 of top 10'),
], loc='lower right')

plt.tight_layout()
plt.savefig('chart5_top_products.png', bbox_inches='tight')
plt.show()

**What I found:** Smartphone, Headphones, and Laptop are the top 3 — all Electronics. Huge gap between top 5 and the rest. This confirms again that Electronics product line is the business's core revenue engine.

## 9. Key Business Insights

Okay so after all this analysis, here's what I'd actually tell a business manager based on the data:

In [ ]:
insights = [
    '1. Electronics is the #1 revenue & profit category — focus inventory and marketing here.',
    '2. All 5 regions contribute almost equally (~20% each) — no one region is being ignored.',
    '3. Discount beyond 10% clearly hurts margins — recommend capping discounts at 10%.',
    '4. Corporate segment has the highest average order value — B2B loyalty programs would help.',
    '5. COD is the top payment method — shows customer hesitation with prepaid; improve trust signals.',
    '6. Smartphone, Headphones, Laptop alone drive a huge share of Electronics revenue.',
    '7. Books & Clothing have thin margins — pushing volume won\'t help much; focus on Electronics.',
    '8. Revenue is consistent year-round — no major seasonal spike, so marketing can be always-on.',
]

print('=' * 58)
print('  KEY BUSINESS INSIGHTS — E-Commerce Sales Analysis 2023')
print('=' * 58)
for ins in insights:
    print(f'  {ins}')
print('=' * 58)

## 10. Summary

This project helped me practice the full EDA pipeline — loading data, cleaning, aggregating, visualising, and deriving insights. The dataset had 5000 orders across 5 categories, 5 regions, and 3 customer segments.

**What I used:**
- `pandas` for all data manipulation and groupby operations
- `matplotlib` + `seaborn` for 5 multi-panel charts
- `pd.cut()` for binning discounts into groups
- `pivot_table()` for the heatmap

**If I extend this further, I'd:**
- Build a profit prediction model using discount + category + region as features
- Make an interactive dashboard in Tableau or Plotly
- Do time-series forecasting for monthly revenue

---
*Tannu Kumari | IGDTUW Delhi | Batch 2026*